# Week 8-D: Measured generator interaction and MIG

Use the canonical W8-C base answers and generation-time semantic routing on FIT and INTERNAL-VAL. No gold answers or promoted answers enter these features.

This notebook computes genuine MiniLM relevance/redundancy/MIG (fixed alpha=beta=0.5), plus teacher-forced **masking attribution**: how much the existing draft's log probability changes when one compressed SFR vector is zeroed before the projector. Positive attribution means support for the existing draft, not that a promotion improves correctness.

**A1–A5 attention measurements are not extracted.** No masking-derived proxies are labelled as attention. A6 is measured draft/passage cosine similarity. Empty drafts receive null masking values and an explicit availability mask.

This is feature extraction only. It does not retrain or change the frozen W8-C selector, rerun W8-A generations, tune on INTERNAL-VAL, or open TEST. A future U4-B comparison requires FIT-only training and a separately frozen evaluation.

In [1]:
import sys, subprocess, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'configs/week8/w8_dafir.yaml').is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
RUN_FULL = True
DEVICE = 'cuda'
OUTPUT = REPO_ROOT / 'results/week8/w8d_attribution/full'
SCRIPT = REPO_ROOT / 'scripts/week8/run_w8d_feature_engineering.py'

def run_extraction(*args):
    # Isolated process frees SARA GPU memory after each split and avoids stale imports.
    subprocess.run([sys.executable, str(SCRIPT), *map(str, args)], cwd=REPO_ROOT, check=True)

print('Python:', sys.executable)
for split in ('fit', 'internal_val'):
    run_extraction('--check', '--split', split)
if not torch.cuda.is_available():
    raise RuntimeError('Select the sara_env kernel on the GPU server.')
free, total = torch.cuda.mem_get_info()
print(f'GPU free: {free / 1024**3:.1f} / {total / 1024**3:.1f} GiB')
if free < 18 * 1024**3:
    raise RuntimeError('Release earlier notebook models to provide at least 18 GiB free GPU memory.')

Python: /home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/bin/python


fit: 1746 queries, 10476 candidates; source fingerprint cb82ce3464718bc3e2e0e260f4f9c779f51c99b08cfc61290b79479bb07eeecf


internal_val: 342 queries, 2052 candidates; source fingerprint 20a74e472187d910397e3faa28c01985eb8e70a024768d24cd6aa29ccce5b1e6


GPU free: 22.8 / 23.5 GiB


## Real FIT smoke check

Two actual FIT queries, one baseline forward pass and six slot-masked forward passes per nonempty draft. Features go to a separate smoke directory. These are not synthetic fixtures or full experimental results.

In [2]:
run_extraction('--smoke', '--split', 'fit', '--smoke-limit', '2', '--device', DEVICE,
               '--output-dir', REPO_ROOT / 'results/week8/w8d_attribution/notebook_smoke')
smoke = pd.read_csv(REPO_ROOT / 'results/week8/w8d_attribution/notebook_smoke/features_fit.csv')
assert len(smoke) == 12
assert not any(c.startswith(('a1_', 'a2_', 'a3_', 'a4_', 'a5_')) for c in smoke)
display(smoke[['question_id', 'semantic_rank', 'candidate_passage_index',
               'compressed_slot_index', 'mig_score', 'masking_available',
               'masking_delta_logp_per_token']])

Saved 12 measured candidate rows to /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8d_attribution/notebook_smoke


,question_id,semantic_rank,candidate_passage_index,compressed_slot_index,mig_score,masking_available,masking_delta_logp_per_token
0,10,5,5,5,0.270818,1,-0.001070
1,10,6,0,0,0.158543,1,-0.002061
2,10,7,2,2,0.155247,1,-0.001620
3,10,8,3,3,0.056655,1,0.001821
4,10,9,1,1,-0.002878,1,-0.003485
5,10,10,4,4,-0.029771,1,-0.000950
6,100,5,7,4,-0.298505,1,-0.039346
7,100,6,4,1,-0.296142,1,-0.002338
8,100,7,3,0,-0.316966,1,-0.020001
9,100,8,6,3,-0.214918,1,-0.003285


## Full FIT extraction

Resumes validated completed queries. Each nonempty draft requires seven teacher-forced SARA forward passes; no new answers are generated. Cache fingerprints reject changed data/checkpoints/code. Extraction time is recorded separately and must be counted in future controller cost comparisons.

In [3]:
if RUN_FULL:
    run_extraction('--full', '--split', 'fit', '--device', DEVICE, '--output-dir', OUTPUT)
else:
    print('Full extraction disabled; smoke outputs remain separate.')

Saved 10476 measured candidate rows to /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8d_attribution/full


## Full INTERNAL-VAL extraction

The feature formula is fixed before this step. Do not choose coefficients, candidate policies, or thresholds from INTERNAL-VAL outcomes.

In [4]:
if RUN_FULL:
    run_extraction('--full', '--split', 'internal_val', '--device', DEVICE, '--output-dir', OUTPUT)

Saved 2052 measured candidate rows to /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8d_attribution/full


## Coverage and feature checks

Unavailable masking values remain missing. Gold likelihood supervision from W8-C is a different target: these draft-attribution features do not backfill it. This notebook makes no final downstream F1 or oracle-gated controller claim.

In [5]:
if RUN_FULL:
    from src.week8.generator_features import FEATURE_COLUMNS, validate_feature_rows, source_fingerprint
    from src.week8.utility_training import load_surface
    for split, expected_queries in [('fit', 1746), ('internal_val', 342)]:
        frame = pd.read_csv(OUTPUT / f'features_{split}.csv', dtype={'question_id': str, 'paper_id': str})
        surface = load_surface(split)
        fingerprint, _ = source_fingerprint(split)
        assert len(frame) == expected_queries * 6
        assert frame.question_id.nunique() == expected_queries
        reference = {str(q): g for q, g in surface.groupby('question_id')}
        for qid, group in frame.groupby('question_id'):
            validate_feature_rows(group.to_dict('records'), reference[qid], fingerprint)
        print(split, '| candidate rows:', len(frame), '| measured masking:', int(frame.masking_available.sum()))
        display(frame[FEATURE_COLUMNS].describe().T)
    print('W8-D features complete. W8-C model and evaluation are unchanged.')

/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


fit | candidate rows: 10476 | measured masking: 10464


,count,mean,std,min,25%,50%,75%,max
query_relevance_rq,10476.0,0.234151,0.147729,-1.101704e-01,0.120230,0.214229,0.340271,0.693314
draft_relevance_ry,10476.0,0.198751,0.172999,-1.606828e-01,0.059775,0.154213,0.317281,0.815677
redundancy_red,10476.0,0.645164,0.140483,-9.782882e-03,0.570786,0.662619,0.742371,1.000000
candidate_diversity,10476.0,0.451073,0.130588,-1.192093e-07,0.360483,0.437492,0.525320,1.029272
mig_score,10476.0,0.010944,0.170660,-4.510056e-01,-0.113820,0.000786,0.121595,0.654042
a6_draft_sim,10476.0,0.198751,0.172999,-1.606828e-01,0.059775,0.154213,0.317281,0.815677
masking_delta_logp,10464.0,-0.017372,0.491577,-6.484421e+00,-0.101845,-0.016792,0.035139,12.359444
masking_delta_logp_per_token,10464.0,-0.004154,0.044638,-5.487294e-01,-0.015080,-0.003308,0.005030,0.837024
masking_target_tokens,10476.0,15.676403,25.245466,0.000000e+00,2.000000,6.000000,18.000000,128.000000
masking_available,10476.0,0.998855,0.033827,0.000000e+00,1.000000,1.000000,1.000000,1.000000


internal_val | candidate rows: 2052 | measured masking: 2052


,count,mean,std,min,25%,50%,75%,max
query_relevance_rq,2052.0,0.234592,0.149204,-8.133502e-02,0.117972,0.209540,0.338852,0.685554
draft_relevance_ry,2052.0,0.182970,0.160181,-1.145074e-01,0.059825,0.132383,0.293630,0.811802
redundancy_red,2052.0,0.644362,0.138472,4.063989e-02,0.564699,0.663833,0.737775,1.000000
candidate_diversity,2052.0,0.446144,0.130238,-1.192093e-07,0.352546,0.436499,0.519494,1.023197
mig_score,2052.0,0.003896,0.170089,-3.964242e-01,-0.116706,-0.012945,0.109526,0.587369
a6_draft_sim,2052.0,0.182970,0.160181,-1.145074e-01,0.059825,0.132383,0.293630,0.811802
masking_delta_logp,2052.0,-0.031144,0.332692,-3.081461e+00,-0.097661,-0.019376,0.027414,5.428926
masking_delta_logp_per_token,2052.0,-0.006981,0.041208,-4.402088e-01,-0.014098,-0.003475,0.003760,0.372121
masking_target_tokens,2052.0,15.479532,25.424299,1.000000e+00,2.000000,6.000000,17.000000,128.000000
masking_available,2052.0,1.000000,0.000000,1.000000e+00,1.000000,1.000000,1.000000,1.000000


W8-D features complete. W8-C model and evaluation are unchanged.
